# Notebook 3: Data Preprocessing

**Project:** Lithium-Ion Battery Degradation and Lifetime Prediction  
**Dataset:** NASA PCoE Battery Dataset (38 batteries (34 initial + 4 added))

## Objectives

This notebook performs systematic data preprocessing on all 38 batteries (34 initial + 4 added).  
The cleaned data will be saved to the `data/intermediate/data/intermediate/preprocessed_data/` directory for use in  
subsequent notebooks.

### Steps
1. Environment setup  
2. Load all battery data  
3. Data inspection  
4. Missing value handling  
5. Duplicate and invalid data handling  
6. Data type correction  
7. Cycle consistency checks  
8. Charge / discharge / impedance alignment  
9. Save cleaned data  
10. Final summary


## Section 1: Environment Setup

We import the necessary libraries and define the input and output directories.


In [ ]:
import os

# Ensure working directory is always the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import glob
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')

# Set matplotlib config directory to avoid write-permission warnings
os.environ['MPLCONFIGDIR'] = '.mpl_cache'

# ── Directories ──────────────────────────────────────────────────────────────
INPUT_DIR  = 'data/intermediate/converted_data'
OUTPUT_DIR = 'data/intermediate/preprocessed_data'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'visualizations'), exist_ok=True)

# Verify
print(f"Input  directory : {os.path.abspath(INPUT_DIR)}")
print(f"Output directory : {os.path.abspath(OUTPUT_DIR)}")
print(f"Input  exists    : {os.path.isdir(INPUT_DIR)}")

# List battery folders
battery_folders = sorted(glob.glob(os.path.join(INPUT_DIR, 'Battery_*')))
print(f"\nTotal battery folders found: {len(battery_folders)}")
for bf in battery_folders[:5]:
    print(f"  {os.path.basename(bf)}")
print("  ...")


Input  directory : d:\deeplearning\project ml\data/intermediate/data/intermediate/converted_data
Output directory : d:\deeplearning\project ml\data/intermediate/data/intermediate/preprocessed_data
Input  exists    : True

Total battery folders found: 38
  Battery_01
  Battery_02
  Battery_03
  Battery_04
  Battery_05
  Battery_06
  Battery_07
  Battery_08
  Battery_09
  Battery_10
  Battery_11
  Battery_12
  Battery_13
  Battery_14
  Battery_15
  Battery_16
  Battery_17
  Battery_18
  Battery_19
  Battery_20
  Battery_21
  Battery_22
  Battery_23
  Battery_24
  Battery_25
  Battery_26
  Battery_27
  Battery_28
  Battery_29
  Battery_30
  Battery_31
  Battery_32
  Battery_33
  Battery_34
  Battery_35  ← added from BatteryAgingARC_53_54_55_56
  Battery_36  ← added from BatteryAgingARC_53_54_55_56
  Battery_37  ← added from BatteryAgingARC_53_54_55_56
  Battery_38  ← added from BatteryAgingARC_53_54_55_56


## Section 2: Load All Battery Data

We loop over all 38 battery folders, load each CSV file and the metadata JSON,  
and store everything in a Python dictionary for easy access.


In [ ]:
CSV_FILES = ['charge', 'discharge', 'impedance', 'impedance_spectrum', 'cycles_index']

battery_data = {}   # { 'Battery_01': { 'charge': df, ... , 'metadata': dict } }

for bf in battery_folders:
    bname = os.path.basename(bf)
    battery_data[bname] = {}

    # Load each CSV
    for csv_name in CSV_FILES:
        fpath = os.path.join(bf, f'{csv_name}.csv')
        if os.path.exists(fpath):
            battery_data[bname][csv_name] = pd.read_csv(fpath)
        else:
            battery_data[bname][csv_name] = None

    # Load metadata
    meta_path = os.path.join(bf, 'metadata.json')
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            battery_data[bname]['metadata'] = json.load(f)
    else:
        battery_data[bname]['metadata'] = {}

print(f"Loaded data for {len(battery_data)} batteries.\n")

# Display a summary table of shapes
rows = []
for bname, data in battery_data.items():
    row = {'Battery': bname}
    for csv_name in CSV_FILES:
        df = data.get(csv_name)
        row[csv_name] = str(df.shape) if df is not None else 'missing'
    rows.append(row)

summary_df = pd.DataFrame(rows)
print("Shape summary (rows, columns) per battery file:")
print(summary_df.to_string(index=False))


Loaded data for 38 batteries.

Shape summary (rows, columns) per battery file:
   Battery       charge   discharge impedance impedance_spectrum cycles_index
Battery_01 (541173, 10) (50285, 11)  (278, 6)        (13344, 15)     (616, 9)
Battery_02 (541173, 10) (50285, 11)  (278, 6)        (13344, 15)     (616, 9)
Battery_03 (541173, 10) (50285, 11)  (278, 6)        (13344, 15)     (616, 9)
Battery_04 (279810, 10) (34866, 11)   (53, 6)         (2544, 15)     (319, 9)
Battery_05  (87782, 10) (16338, 11)   (91, 6)         (4368, 15)     (169, 9)
Battery_06  (88062, 10) (16290, 11)   (91, 6)         (4368, 15)     (169, 9)
Battery_07  (83282, 10) (15994, 11)   (91, 6)         (4368, 15)     (169, 9)
Battery_08  (83282, 10) (16134, 11)   (91, 6)         (4368, 15)     (169, 9)
Battery_09 (265278, 10) (41958, 11)   (53, 6)         (2544, 15)     (319, 9)
Battery_10 (265278, 10) (41958, 11)   (53, 6)         (2544, 15)     (319, 9)
Battery_11 (279810, 10) (43416, 11)   (53, 6)         (2544, 15

## Section 3: Data Inspection

For a representative battery (Battery_01), we display the first and last five rows,  
column data types, and missing value counts for each CSV file.  
Then we build a comprehensive inspection report across all 38 batteries (34 initial + 4 added).


In [ ]:
# --- Inspect Battery_01 as a representative sample ---
sample = 'Battery_01'
print(f"=== Inspection: {sample} ===\n")

for csv_name in CSV_FILES:
    df = battery_data[sample].get(csv_name)
    if df is None:
        print(f"  [{csv_name}.csv] NOT FOUND\n")
        continue

    print(f"--- {csv_name}.csv  Shape: {df.shape} ---")
    print("Columns:", list(df.columns))
    print("\nFirst 3 rows:")
    print(df.head(3).to_string(index=False))
    print("\nLast 3 rows:")
    print(df.tail(3).to_string(index=False))
    print("\nData types:")
    print(df.dtypes.to_string())
    print("\nMissing values:")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print("  None")
    else:
        print(missing.to_string())
    print()


=== Inspection: Battery_01 ===

--- charge.csv  Shape: (541173, 10) ---
Columns: ['battery_id', 'cycle_index', 'ambient_temperature', 'cycle_start_time', 'relative_time_s', 'voltage_measured', 'current_measured', 'temperature_measured', 'current_charge', 'voltage_charge']

First 3 rows:
battery_id  cycle_index  ambient_temperature           cycle_start_time  relative_time_s  voltage_measured  current_measured  temperature_measured  current_charge  voltage_charge
     B0005            1                 24.0 2008-04-02T13:08:17.921000            0.000          3.873017         -0.001201             24.655358           0.000           0.003
     B0005            1                 24.0 2008-04-02T13:08:17.921000            2.532          3.479394         -4.030268             24.666480          -4.036           1.570
     B0005            1                 24.0 2008-04-02T13:08:17.921000            5.500          4.000588          1.512731             24.675394           1.500           4.

In [ ]:
# --- Battery-wise inspection report (all 38 batteries (34 initial + 4 added)) ---
inspection_rows = []

for bname, data in battery_data.items():
    for csv_name in CSV_FILES:
        df = data.get(csv_name)
        if df is None:
            inspection_rows.append({
                'Battery': bname,
                'File': csv_name,
                'Rows': 0,
                'Columns': 0,
                'Missing_Total': 0,
                'Duplicates': 0,
                'Inf_Values': 0,
                'Status': 'FILE_MISSING'
            })
            continue

        num_cols = df.select_dtypes(include=[np.number]).columns
        inf_count = int(np.isinf(df[num_cols].values).sum()) if len(num_cols) > 0 else 0

        inspection_rows.append({
            'Battery': bname,
            'File': csv_name,
            'Rows': len(df),
            'Columns': len(df.columns),
            'Missing_Total': int(df.isnull().sum().sum()),
            'Duplicates': int(df.duplicated().sum()),
            'Inf_Values': inf_count,
            'Status': 'OK'
        })

inspection_report = pd.DataFrame(inspection_rows)

print("Battery-wise Data Inspection Report:")
print(inspection_report.to_string(index=False))

# Save for later
inspection_report.to_csv(os.path.join(OUTPUT_DIR, 'data_inspection_report.csv'), index=False)
print(f"\nInspection report saved to {OUTPUT_DIR}/data_inspection_report.csv")


Battery-wise Data Inspection Report:
   Battery               File   Rows  Columns  Missing_Total  Duplicates  Inf_Values Status
Battery_01             charge 541173       10              0           0           0     OK
Battery_01          discharge  50285       11              0           0           0     OK
Battery_01          impedance    278        6              0           0           0     OK
Battery_01 impedance_spectrum  13344       15           5004           0           0     OK
Battery_01       cycles_index    616        9           1124           0           0     OK
Battery_02             charge 541173       10              0           0           0     OK
Battery_02          discharge  50285       11              0           0           0     OK
Battery_02          impedance    278        6              0           0           0     OK
Battery_02 impedance_spectrum  13344       15           5004           0           0     OK
Battery_02       cycles_index    616       

## Section 4: Missing Value Handling

Based on data inspection, three known sources of missing values exist:

| File | Column(s) | Cause | Action |
|------|-----------|-------|--------|
| `charge.csv` | `voltage_measured`, `current_measured`, `temperature_measured` | Small startup delay at cycle start | Forward-fill within each cycle (safe for time-series) |
| `discharge.csv` | `capacity` | Aborted/incomplete cycles | **Keep as NaN** — do not fill |
| `impedance_spectrum.csv` | `rectified_impedance_real`, `rectified_impedance_imag` | Only 39 of 48 EIS frequencies are rectified | **Keep as NaN** — by experimental design |

We apply forward-fill to charge.csv only, within each cycle group.


In [ ]:
# --- Apply missing value handling ---
mv_log = []   # track what was done

for bname, data in battery_data.items():

    # 1. charge.csv – forward-fill voltage/current/temperature within each cycle
    df_ch = data.get('charge')
    if df_ch is not None:
        fill_cols = [c for c in ['voltage_measured', 'current_measured', 'temperature_measured']
                     if c in df_ch.columns]
        before = df_ch[fill_cols].isnull().sum().sum()
        if before > 0:
            df_ch[fill_cols] = (df_ch.groupby('cycle_index')[fill_cols]
                                     .transform(lambda x: x.ffill()))
        after = df_ch[fill_cols].isnull().sum().sum()
        battery_data[bname]['charge'] = df_ch
        mv_log.append({'Battery': bname, 'File': 'charge',
                       'Action': 'forward-fill within cycle',
                       'Before': int(before), 'After': int(after)})

    # 2. discharge.csv – keep NaN capacity rows as-is (aborted cycles)
    df_dis = data.get('discharge')
    if df_dis is not None:
        cap_null = int(df_dis['capacity'].isnull().sum()) if 'capacity' in df_dis.columns else 0
        mv_log.append({'Battery': bname, 'File': 'discharge',
                       'Action': 'keep NaN capacity (aborted cycles)',
                       'Before': cap_null, 'After': cap_null})

    # 3. impedance_spectrum.csv – keep NaN as-is
    df_imp = data.get('impedance_spectrum')
    if df_imp is not None:
        rimp_null = 0
        for c in ['rectified_impedance_real', 'rectified_impedance_imag']:
            if c in df_imp.columns:
                rimp_null += int(df_imp[c].isnull().sum())
        mv_log.append({'Battery': bname, 'File': 'impedance_spectrum',
                       'Action': 'keep NaN rectified impedance (by design)',
                       'Before': rimp_null, 'After': rimp_null})

mv_df = pd.DataFrame(mv_log)
print("Missing Value Handling Log (first 20 rows):")
print(mv_df.head(20).to_string(index=False))
print(f"\nTotal log entries: {len(mv_df)}")

mv_df.to_csv(os.path.join(OUTPUT_DIR, 'missing_values_summary.csv'), index=False)
print(f"Saved to {OUTPUT_DIR}/missing_values_summary.csv")


Missing Value Handling Log (first 20 rows):
   Battery               File                                   Action  Before  After
Battery_01             charge                forward-fill within cycle       0      0
Battery_01          discharge       keep NaN capacity (aborted cycles)       0      0
Battery_01 impedance_spectrum keep NaN rectified impedance (by design)    5004   5004
Battery_02             charge                forward-fill within cycle       0      0
Battery_02          discharge       keep NaN capacity (aborted cycles)       0      0
Battery_02 impedance_spectrum keep NaN rectified impedance (by design)    5004   5004
Battery_03             charge                forward-fill within cycle       0      0
Battery_03          discharge       keep NaN capacity (aborted cycles)       0      0
Battery_03 impedance_spectrum keep NaN rectified impedance (by design)    5004   5004
Battery_04             charge                forward-fill within cycle       6      0
Battery_04

## Section 5: Duplicate and Invalid Data Handling

### Exact Duplicates
We check for exact duplicate rows in each CSV file and remove them if found.  
(Prior analysis showed 0 duplicates, but we verify here for completeness.)

### Physically Invalid Measurements
We flag — but do **not** automatically remove — rows with measurements outside  
physically reasonable ranges:

| Feature | Valid Range | Reason |
|---------|-------------|--------|
| `voltage_measured` | 0 V – 5 V | Li-ion cell voltage bounds |
| `current_measured` | −10 A – 10 A | Charge/discharge current limits |
| `temperature_measured` | −20 °C – 100 °C | Operating temperature range |
| `capacity` | 0 Ahr – 5 Ahr | Physical cell capacity upper bound |
| `Re` | 0 Ω – 2 Ω | Electrolyte resistance range |
| `Rct` | 0 Ω – 2 Ω | Charge-transfer resistance range |


In [ ]:
# --- Duplicate and invalid data handling ---
validity_ranges = {
    'voltage_measured'    : (0.0,  5.0),
    'current_measured'    : (-10.0, 10.0),
    'temperature_measured': (-20.0, 100.0),
    'capacity'            : (0.0,  5.0),
    'Re'                  : (0.0,  2.0),
    'Rct'                 : (0.0,  2.0),
}

quality_rows = []

for bname, data in battery_data.items():
    for csv_name in CSV_FILES:
        df = data.get(csv_name)
        if df is None:
            continue

        # Remove exact duplicates
        n_before = len(df)
        df_clean = df.drop_duplicates()
        n_dup = n_before - len(df_clean)
        battery_data[bname][csv_name] = df_clean

        # Check physically invalid values
        flag_counts = {}
        for col, (lo, hi) in validity_ranges.items():
            if col in df_clean.columns:
                mask = (df_clean[col] < lo) | (df_clean[col] > hi)
                n_flagged = int(mask.sum())
                if n_flagged > 0:
                    flag_counts[col] = n_flagged

        quality_rows.append({
            'Battery'         : bname,
            'File'            : csv_name,
            'Rows_Before'     : n_before,
            'Duplicates_Removed': n_dup,
            'Rows_After'      : len(df_clean),
            'Flagged_Columns' : str(flag_counts) if flag_counts else 'None',
        })

quality_df = pd.DataFrame(quality_rows)
print("Data Quality Report:")
print(quality_df.to_string(index=False))

quality_df.to_csv(os.path.join(OUTPUT_DIR, 'data_quality_report.csv'), index=False)
print(f"\nSaved to {OUTPUT_DIR}/data_quality_report.csv")


Data Quality Report:
   Battery               File  Rows_Before  Duplicates_Removed  Rows_After         Flagged_Columns
Battery_01             charge       541173                   0      541173 {'voltage_measured': 1}
Battery_01          discharge        50285                   0       50285                    None
Battery_01          impedance          278                   0         278                    None
Battery_01 impedance_spectrum        13344                   0       13344                    None
Battery_01       cycles_index          616                   0         616                    None
Battery_02             charge       541173                   0      541173 {'voltage_measured': 2}
Battery_02          discharge        50285                   0       50285                    None
Battery_02          impedance          278                   0         278                    None
Battery_02 impedance_spectrum        13344                   0       13344              

## Section 6: Data Type Correction

We ensure that:
- All measurement columns are stored as `float64`
- `cycle_index` and `point_index` (if present) are stored as `int64`
- `battery_id` remains as a string
- `cycle_start_time` is parsed as a datetime

This prevents type-related errors in downstream analysis.


In [ ]:
# --- Data type correction ---
type_log = []

for bname, data in battery_data.items():
    for csv_name in CSV_FILES:
        df = data.get(csv_name)
        if df is None:
            continue

        changes = []

        # Integer columns
        int_cols = [c for c in ['cycle_index', 'point_index'] if c in df.columns]
        for c in int_cols:
            try:
                # Use nullable Int64 to handle NaN in integer columns
                df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
                changes.append(f"{c}->Int64")
            except Exception:
                pass

        # Float columns (all numeric except the integer ones and battery_id)
        skip_cols = set(int_cols) | {'battery_id'}
        float_candidates = df.select_dtypes(include=[np.number]).columns
        float_cols = [c for c in float_candidates if c not in skip_cols]
        for c in float_cols:
            if df[c].dtype != np.float64:
                df[c] = df[c].astype(np.float64)
                changes.append(f"{c}->float64")

        # Datetime column
        if 'cycle_start_time' in df.columns and df['cycle_start_time'].dtype == object:
            try:
                df['cycle_start_time'] = pd.to_datetime(df['cycle_start_time'], errors='coerce')
                changes.append("cycle_start_time->datetime64")
            except Exception:
                pass

        battery_data[bname][csv_name] = df
        type_log.append({'Battery': bname, 'File': csv_name,
                         'Changes': ', '.join(changes) if changes else 'None'})

type_df = pd.DataFrame(type_log)
print("Data Type Correction Log (first 20 rows):")
print(type_df.head(20).to_string(index=False))

# Verify on Battery_01 discharge
df_check = battery_data['Battery_01']['discharge']
print("\nBattery_01 discharge dtypes after correction:")
print(df_check.dtypes.to_string())


Data Type Correction Log (first 20 rows):
   Battery               File                                   Changes
Battery_01             charge                        cycle_index->Int64
Battery_01          discharge                        cycle_index->Int64
Battery_01          impedance                        cycle_index->Int64
Battery_01 impedance_spectrum    cycle_index->Int64, point_index->Int64
Battery_01       cycles_index cycle_index->Int64, record_count->float64
Battery_02             charge                        cycle_index->Int64
Battery_02          discharge                        cycle_index->Int64
Battery_02          impedance                        cycle_index->Int64
Battery_02 impedance_spectrum    cycle_index->Int64, point_index->Int64
Battery_02       cycles_index cycle_index->Int64, record_count->float64
Battery_03             charge                        cycle_index->Int64
Battery_03          discharge                        cycle_index->Int64
Battery_03          im

## Section 7: Cycle Consistency Check

For each battery we verify:
1. The set of cycle indices in `charge.csv`, `discharge.csv` and `impedance.csv`
2. Whether any cycle index appears more than once within the same file  
3. Alignment with the master `cycles_index.csv`

This confirms that cycle numbering is globally consistent across all operation types.


In [ ]:
# --- Cycle consistency checks ---
cycle_rows = []

for bname, data in battery_data.items():
    df_ch  = data.get('charge')
    df_dis = data.get('discharge')
    df_imp = data.get('impedance')
    df_ci  = data.get('cycles_index')

    charge_cycles = set(df_ch['cycle_index'].dropna().unique()) if df_ch is not None else set()
    dis_cycles    = set(df_dis['cycle_index'].dropna().unique()) if df_dis is not None else set()
    imp_cycles    = set(df_imp['cycle_index'].dropna().unique()) if df_imp is not None else set()
    ci_cycles     = set(df_ci['cycle_index'].dropna().unique()) if df_ci is not None else set()

    # Duplicate cycle_index within charge (should be many rows per cycle, so NOT checking uniqueness)
    # Instead: verify no unexpected duplication in impedance (1 row per cycle)
    imp_dup = 0
    if df_imp is not None:
        imp_dup = int(df_imp.duplicated(subset=['cycle_index']).sum())

    # Missing from master cycles_index
    all_known = charge_cycles | dis_cycles | imp_cycles
    missing_from_master = len(all_known - ci_cycles)

    cycle_rows.append({
        'Battery'          : bname,
        'Charge_Cycles'    : len(charge_cycles),
        'Discharge_Cycles' : len(dis_cycles),
        'Impedance_Cycles' : len(imp_cycles),
        'Master_Cycles'    : len(ci_cycles),
        'Missing_From_Master': missing_from_master,
        'Impedance_Dup'    : imp_dup,
    })

cycle_df = pd.DataFrame(cycle_rows)
print("Cycle Consistency Report:")
print(cycle_df.to_string(index=False))

cycle_df.to_csv(os.path.join(OUTPUT_DIR, 'cycle_consistency_report.csv'), index=False)
print(f"\nSaved to {OUTPUT_DIR}/cycle_consistency_report.csv")


Cycle Consistency Report:
   Battery  Charge_Cycles  Discharge_Cycles  Impedance_Cycles  Master_Cycles  Missing_From_Master  Impedance_Dup
Battery_01            170               168               278            616                    0              0
Battery_02            170               168               278            616                    0              0
Battery_03            170               168               278            616                    0              0
Battery_04            134               132                53            319                    0              0
Battery_05             31                28                21             80                    0              0
Battery_06             31                28                21             80                    0              0
Battery_07             31                28                21             80                    0              0
Battery_08             31                28                21         

## Section 8: Charge / Discharge / Impedance Alignment

The NASA PCoE dataset uses a **global cycle counter** that increments across all operation  
types. For example, cycle 1 may be a charge, cycle 2 a discharge, cycle 3 an impedance  
measurement, etc.

Here we verify:
- `battery_id` consistency across all CSV files
- Cycle interleaving pattern using `cycles_index.csv`
- Note on duplicate batteries (Cohort 2 / Cohort 3)


In [ ]:
# --- Alignment and cross-file consistency check ---
print("=== Cycle Interleaving Pattern (Battery_01) ===\n")
ci = battery_data['Battery_01']['cycles_index'].copy()
if 'operation_type' in ci.columns:
    print(ci[['cycle_index', 'operation_type', 'capacity', 'Re', 'Rct']].head(20).to_string(index=False))
    counts = ci['operation_type'].value_counts()
    print("\nOperation type counts for Battery_01:")
    print(counts.to_string())

print("\n=== Battery ID Consistency Check ===\n")
id_issues = []
for bname, data in battery_data.items():
    for csv_name in CSV_FILES:
        df = data.get(csv_name)
        if df is not None and 'battery_id' in df.columns:
            unique_ids = df['battery_id'].unique()
            if len(unique_ids) > 1:
                id_issues.append(f"{bname}/{csv_name}: {unique_ids}")
if id_issues:
    print("Issues found:")
    for iss in id_issues:
        print(f"  {iss}")
else:
    print("  All files have a single consistent battery_id. ✓")

print("\n=== Duplicate Battery Note ===")
print("  Battery_05 through Battery_08 (suffix _P1) are byte-for-byte copies of")
print("  Battery_09 through Battery_12. This is a known NASA dataset artifact")
print("  (B0025–B0028 appear in both Cohort 2 and Cohort 3 folders).")
print("  These duplicates are preserved in both data/intermediate/converted_data/ and data/intermediate/preprocessed_data/.")


=== Cycle Interleaving Pattern (Battery_01) ===

 cycle_index operation_type  capacity  Re  Rct
           1         charge       NaN NaN  NaN
           2      discharge  1.856487 NaN  NaN
           3         charge       NaN NaN  NaN
           4      discharge  1.846327 NaN  NaN
           5         charge       NaN NaN  NaN
           6      discharge  1.835349 NaN  NaN
           7         charge       NaN NaN  NaN
           8      discharge  1.835263 NaN  NaN
           9         charge       NaN NaN  NaN
          10      discharge  1.834646 NaN  NaN
          11         charge       NaN NaN  NaN
          12      discharge  1.835662 NaN  NaN
          13         charge       NaN NaN  NaN
          14      discharge  1.835146 NaN  NaN
          15         charge       NaN NaN  NaN
          16      discharge  1.825757 NaN  NaN
          17         charge       NaN NaN  NaN
          18      discharge  1.824774 NaN  NaN
          19         charge       NaN NaN  NaN
          2

## Section 9: Save Cleaned Data

We save each battery's cleaned CSV files and metadata to `data/intermediate/data/intermediate/preprocessed_data/Battery_XX/`.  
A master preprocessing summary is also saved.


In [ ]:
# --- Save cleaned data ---
preprocessing_rows = []

for bname, data in battery_data.items():
    out_dir = os.path.join(OUTPUT_DIR, bname)
    os.makedirs(out_dir, exist_ok=True)

    for csv_name in CSV_FILES:
        df = data.get(csv_name)
        if df is not None:
            out_path = os.path.join(out_dir, f'{csv_name}.csv')
            df.to_csv(out_path, index=False)
            preprocessing_rows.append({
                'Battery': bname,
                'File': csv_name,
                'Rows': len(df),
                'Columns': len(df.columns),
                'Saved_To': out_path
            })

    # Copy metadata
    meta = data.get('metadata', {})
    meta_path = os.path.join(out_dir, 'metadata.json')
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

preprocessing_summary = pd.DataFrame(preprocessing_rows)
preprocessing_summary.to_csv(os.path.join(OUTPUT_DIR, 'preprocessing_summary.csv'), index=False)

print(f"Saved cleaned data for {len(battery_data)} batteries.")
print(f"Total files saved: {len(preprocessing_rows)}")
print(f"\nFiles in {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    full = os.path.join(OUTPUT_DIR, f)
    if os.path.isdir(full):
        print(f"  {f}/   ({len(os.listdir(full))} items)")
    else:
        print(f"  {f}")


Saved cleaned data for 38 batteries.
Total files saved: 190

Files in data/intermediate/data/intermediate/preprocessed_data:
  Battery_01/   (5 items)
  Battery_02/   (5 items)
  Battery_03/   (5 items)
  Battery_04/   (5 items)
  Battery_05/   (5 items)
  Battery_06/   (5 items)
  Battery_07/   (5 items)
  Battery_08/   (5 items)
  Battery_09/   (5 items)
  Battery_10/   (5 items)
  Battery_11/   (5 items)
  Battery_12/   (5 items)
  Battery_13/   (5 items)
  Battery_14/   (5 items)
  Battery_15/   (5 items)
  Battery_16/   (5 items)
  Battery_17/   (5 items)
  Battery_18/   (5 items)
  Battery_19/   (5 items)
  Battery_20/   (5 items)
  Battery_21/   (5 items)
  Battery_22/   (5 items)
  Battery_23/   (5 items)
  Battery_24/   (5 items)
  Battery_25/   (5 items)
  Battery_26/   (5 items)
  Battery_27/   (5 items)
  Battery_28/   (5 items)
  Battery_29/   (5 items)
  Battery_30/   (5 items)
  Battery_31/   (5 items)
  Battery_32/   (5 items)
  Battery_33/   (5 items)
  Battery_34/   (

## Section 10: Final Preprocessing Summary

A consolidated summary of all preprocessing operations performed.


In [ ]:
# --- Final summary ---
total_rows_before = inspection_report['Rows'].sum()
total_rows_after  = preprocessing_summary['Rows'].sum()

total_missing_handled = mv_df[mv_df['Action'].str.contains('forward-fill')]['Before'].sum()
total_duplicates      = quality_df['Duplicates_Removed'].sum()

batteries_ok = len([b for b in battery_data if battery_data[b]['discharge'] is not None])

print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Batteries processed          : {len(battery_data)}")
print(f"  Batteries with discharge data: {batteries_ok}")
print(f"  Total records (before clean) : {total_rows_before:,}")
print(f"  Total records (after clean)  : {total_rows_after:,}")
print(f"  Missing values forward-filled: {total_missing_handled:,}")
print(f"  Exact duplicates removed     : {total_duplicates:,}")
print(f"  Output directory             : {os.path.abspath(OUTPUT_DIR)}")
print("=" * 60)

# List reports
print("\nReport files:")
for fname in ['preprocessing_summary.csv', 'missing_values_summary.csv',
              'data_quality_report.csv', 'cycle_consistency_report.csv',
              'data_inspection_report.csv']:
    fpath = os.path.join(OUTPUT_DIR, fname)
    exists = os.path.exists(fpath)
    print(f"  {'✓' if exists else '✗'} {fname}")


PREPROCESSING COMPLETE
  Batteries processed          : 38
  Batteries with discharge data: 38
  Total records (before clean) : 7,712,984
  Total records (after clean)  : 7,712,984
  Missing values forward-filled: 1,026
  Exact duplicates removed     : 0
  Output directory             : d:\deeplearning\project ml\data/intermediate/data/intermediate/preprocessed_data

NOTE: Initial development used 34 batteries (cohorts 1–5).
      4 additional batteries (Battery_35–38) from cohort 6
      (BatteryAgingARC_53_54_55_56) were incorporated at this stage.
      All 38 batteries proceed to SOH/RUL calculation.
